# Best Epoch vs Last Epoch Comparison

Quick sanity check: evaluate `model_best.pt` (best validation checkpoint) vs `model_epoch###.pt` (last epoch) on the same period. Confirms best-checkpoint selection isn't drastically different from last-epoch behavior.

In [ ]:
# ── CONFIG: change these to point at any run ──
import sys, os
sys.path.insert(0, os.path.abspath("../../"))

from pathlib import Path

# Point to the run directory containing model_best.pt and model_epoch###.pt
BASIN = "guerneville"
RUN_LABEL = "CROSS_VAL_V5"
RUN_STAMP = "20260303T045131Z"

# Which sub-run to compare (e.g. testing_run_0303_085627)
# Set to None to auto-pick the latest
SUB_RUN = None

# Evaluation period: "validation" or "test"
PERIOD = "test"

# MTS model?
IS_MTS = True

METRICS = ["NSE", "MSE", "KGE", "Pearson-r"]

In [ ]:
# ── Resolve run directory ──
from neuralhydrology.evaluation.tester import RegressionTester
from neuralhydrology.utils.config import Config
import pandas as pd
import torch

base = Path(f"../../outputs/{BASIN}/mts_shared/runs/{RUN_LABEL}_{RUN_STAMP}")
if not base.exists():
    # Try non-MTS path
    base = Path(f"../../outputs/{BASIN}/daily/{RUN_LABEL}_{RUN_STAMP}")

if SUB_RUN:
    run_dir = base / SUB_RUN
else:
    # Pick latest testing_run_* directory
    runs = sorted([d for d in base.iterdir() if d.is_dir() and d.name.startswith("testing_run_")])
    run_dir = runs[-1]

print(f"Run dir: {run_dir.name}")

# Read best/last epoch info
best_epoch = int((run_dir / "best_epoch.txt").read_text().strip())
last_epoch_file = sorted(run_dir.glob("model_epoch*.pt"))[-1]
last_epoch = int(last_epoch_file.stem.replace("model_epoch", ""))
gap = last_epoch - best_epoch

print(f"Best epoch: {best_epoch}")
print(f"Last epoch: {last_epoch}")
print(f"Gap: {gap} epochs")

In [ ]:
# ── Evaluate both checkpoints ──
import logging
logging.getLogger("neuralhydrology").setLevel(logging.WARNING)

cfg = Config(run_dir / "config.yml")
cfg._cfg["device"] = "cpu"
cfg._cfg["run_dir"] = str(run_dir)

def eval_checkpoint(weight_path, period, metrics_list):
    """Evaluate a specific weight file and return metrics dict(s)."""
    tester = RegressionTester(cfg=cfg, run_dir=run_dir, period=period, init_model=True)
    tester.model.load_state_dict(torch.load(weight_path, map_location="cpu"))
    results = tester.evaluate(save_results=False, metrics=metrics_list, model=tester.model)
    # Extract metrics from xarray results
    basin_key = list(results.keys())[0]
    ds = results[basin_key]
    out = {}
    if IS_MTS:
        for freq in ["1D", "1H"]:
            freq_ds = ds.sel(frequency=freq) if "frequency" in ds.dims else ds
            for m in metrics_list:
                key = f"{m}_{freq}"
                if m in freq_ds:
                    out[key] = float(freq_ds[m].values)
    else:
        for m in metrics_list:
            if m in ds:
                out[m] = float(ds[m].values)
    return out

print(f"Evaluating best checkpoint (epoch {best_epoch})...")
best_metrics = eval_checkpoint(run_dir / "model_best.pt", PERIOD, METRICS)

print(f"Evaluating last checkpoint (epoch {last_epoch})...")
last_metrics = eval_checkpoint(last_epoch_file, PERIOD, METRICS)

print("Done.")

In [ ]:
# ── Results table ──
rows = []
for key in sorted(best_metrics.keys()):
    b = best_metrics[key]
    l = last_metrics[key]
    delta = b - l
    rows.append({"Metric": key, f"Best (ep {best_epoch})": f"{b:.6f}", f"Last (ep {last_epoch})": f"{l:.6f}", "Delta (best-last)": f"{delta:+.6f}"})

df_compare = pd.DataFrame(rows)
print(f"\n{'='*70}")
print(f"  {BASIN.upper()} | {RUN_LABEL} | {PERIOD} | {run_dir.name}")
print(f"  Best epoch: {best_epoch} | Last epoch: {last_epoch} | Gap: {gap}")
print(f"{'='*70}")
print(df_compare.to_string(index=False))
print(f"{'='*70}")
print("Positive delta = best checkpoint is better (expected).")

In [ ]:
# ── Compare ALL sub-runs in this experiment ──
all_rows = []
sub_runs = sorted([d for d in base.iterdir() if d.is_dir() and d.name.startswith("testing_run_")])

for sr in sub_runs:
    best_ep_file = sr / "best_epoch.txt"
    last_ep_files = sorted(sr.glob("model_epoch*.pt"))
    if not best_ep_file.exists() or not last_ep_files:
        continue
    
    b_ep = int(best_ep_file.read_text().strip())
    l_ep_file = last_ep_files[-1]
    l_ep = int(l_ep_file.stem.replace("model_epoch", ""))
    
    # Determine period from results files
    has_test = any(sr.glob("results_output_test_*.csv"))
    has_val = any(sr.glob("results_output_validation_*.csv"))
    period = "test" if has_test else "validation"
    
    cfg_sr = Config(sr / "config.yml")
    cfg_sr._cfg["device"] = "cpu"
    cfg_sr._cfg["run_dir"] = str(sr)
    
    try:
        bm = eval_checkpoint(sr / "model_best.pt", period, ["NSE"])
        lm = eval_checkpoint(l_ep_file, period, ["NSE"])
    except Exception as e:
        print(f"  Skipping {sr.name}: {e}")
        continue
    
    row = {"sub_run": sr.name, "period": period, "best_ep": b_ep, "last_ep": l_ep, "gap": l_ep - b_ep}
    for k in sorted(bm.keys()):
        row[f"{k}_best"] = bm[k]
        row[f"{k}_last"] = lm[k]
        row[f"{k}_delta"] = bm[k] - lm[k]
    all_rows.append(row)

df_all = pd.DataFrame(all_rows)
print(f"\n{'='*90}")
print(f"  ALL SUB-RUNS: {BASIN.upper()} | {RUN_LABEL}")
print(f"{'='*90}")
print(df_all.to_string(index=False))
print(f"\nPositive delta = best checkpoint outperforms last epoch.")